# Data Preprocessing - Sheffield Road Collision Dataset

This notebook covers the complete preprocessing pipeline for the "Filtered_Sheffield_Traffic_Data.csv". The pipeline is structured as follows:
1.

In [3]:
# Install the relevant packages in the environment this notebook runs in
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn scikit-learn scipy


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

CONFIG = {
    'data_path': "../raw/Filtered_Sheffield_Traffic_Data.csv",
    'output_path': "../processed/Sheffield_Traffic_Proccesed.csv",
    
    # STATS19 sentinel value meaning "unknown / not applicable"
    'sentinel_value': -1,

    # Columns that are identifiers or administrative -- not useful for ML
    'drop_cols': [
        'collision_ref_no',          # redundant with collision_index
        'police_force',              # constant (always 14 = South Yorkshire)
        'local_authority_district',  # constant (always 215)
        'local_authority_ons_district',  # near-constant
        'local_authority_highway',   # near-constant
        'junction_detail_historic',  # superseded by junction_detail
        'pedestrian_crossing_human_control_historic',   # superseded
        'pedestrian_crossing_physical_facilities_historic',  # superseded
        'carriageway_hazards_historic',  # superseded by carriageway_hazards
        'enhanced_severity_collision',   # almost entirely -1 (7322/7933)
        'collision_adjusted_severity_serious',  # >5000 missing
        'collision_adjusted_severity_slight',   # >5000 missing
        'lsoa_of_accident_location',  # high-cardinality string ID
    ],

    # Categorical columns encoded with sentinel -1 -- replace with NaN then impute by mode
    'sentinel_categorical_cols': [
        'junction_control',
        'second_road_class',
        'pedestrian_crossing',
        'road_surface_conditions',
        'special_conditions_at_site',
        'carriageway_hazards',
        'urban_or_rural_area',
        'trunk_road_flag',
        'collision_injury_based',
    ],

    # Cols where -1 should just be treated as a valid "unknown" category (not imputed)
    'sentinel_keep_as_category': [
        'did_police_officer_attend_scene_of_accident',  # -1 = unknown, meaningful
        'second_road_number',  # -1 = no second road, meaningful
    ],

    # Columns needing KNN imputation (continuous coords with NaN)
    'knn_impute_cols': ['location_easting_osgr', 'location_northing_osgr'],
    'knn_neighbours': 5,

    # Continuous columns to check for outliers via IQR
    'outlier_cols': [
        'number_of_vehicles',
        'number_of_casualties',
        'location_easting_osgr',
        'location_northing_osgr',
    ],
    'iqr_multiplier': 1.5,

    # Columns to min-max scale (continuous numeric features)
    'scale_cols': [
        'location_easting_osgr',
        'location_northing_osgr',
        'number_of_vehicles',
        'number_of_casualties',
        'collision_year',
    ],

    # Categorical columns to label-encode for ML readiness
    'encode_cols': [
        'day_of_week', 'road_type', 'speed_limit', 'junction_detail',
        'junction_control', 'second_road_class', 'light_conditions',
        'weather_conditions', 'road_surface_conditions',
        'special_conditions_at_site', 'carriageway_hazards',
        'urban_or_rural_area', 'first_road_class',
    ],

    # Plot styling
    'figsize_wide': (16, 6),
    'figsize_square': (10, 8),
    'palette': 'Set2',
    'random_state': 42,
}

sns.set_theme(style='whitegrid', palette=CONFIG['palette'])
print('Config loaded. Libraries imported.')

Config loaded. Libraries imported.


In [5]:
df_raw = pd.read_csv(CONFIG['data_path'], encoding='utf-8-sig')

print(f'Shape: {df_raw.shape}')
print(f'Column dtypes: {df_raw.dtypes}')
print(f'First 3 rows:')
df_raw.head(3)

df_raw.describe()

Shape: (7933, 44)
Column dtypes: collision_index                                         str
collision_year                                        int64
collision_ref_no                                        str
location_easting_osgr                               float64
location_northing_osgr                              float64
longitude                                           float64
latitude                                            float64
police_force                                          int64
collision_severity                                    int64
number_of_vehicles                                    int64
number_of_casualties                                  int64
date                                                    str
day_of_week                                           int64
time                                                    str
local_authority_district                              int64
local_authority_ons_district                            str
local_a

,collision_year,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,number_of_casualties,day_of_week,...,special_conditions_at_site,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight
count,7933.000000,7868.000000,7868.000000,3693.000000,3693.000000,7933.0,7933.000000,7933.000000,7933.000000,7933.000000,...,7933.000000,7933.000000,7933.000000,7933.000000,7933.000000,7933.000000,7933.000000,7933.000000,2696.000000,2696.000000
mean,1997.699987,435874.978012,388616.385231,-1.460544,53.385982,14.0,2.813185,1.696080,1.300139,4.195260,...,0.097819,0.062145,0.758855,0.279970,-0.307702,0.795286,-0.623598,-0.592714,0.235354,0.727925
std,11.405115,3611.910193,21711.218888,0.057606,0.036132,0.0,0.420590,0.680231,0.740409,1.925508,...,0.623956,0.433466,3.439919,1.069283,1.074735,1.456096,1.378463,0.628883,0.336737,0.362611
min,1979.000000,340780.000000,338800.000000,-2.891207,52.945371,14.0,1.000000,1.000000,1.000000,1.000000,...,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0.000000,0.000000
25%,1988.000000,434169.000000,385300.000000,-1.485380,53.364713,14.0,3.000000,1.000000,1.000000,3.000000,...,0.000000,0.000000,0.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0.019863,0.729183
50%,1997.000000,435590.000000,387358.000000,-1.464603,53.382543,14.0,3.000000,2.000000,1.000000,4.000000,...,0.000000,0.000000,0.000000,1.000000,-1.000000,2.000000,-1.000000,-1.000000,0.091170,0.892594
75%,2007.000000,437725.500000,390050.000000,-1.430020,53.407034,14.0,3.000000,2.000000,1.000000,6.000000,...,0.000000,0.000000,0.000000,1.000000,1.000000,2.000000,-1.000000,0.000000,0.242816,0.972434
max,2020.000000,487490.000000,984700.000000,-1.328944,54.247744,14.0,3.000000,10.000000,11.000000,7.000000,...,7.000000,7.000000,21.000000,3.000000,2.000000,2.000000,7.000000,1.000000,1.000000,1.000000


In [ ]:
# True NaN missing values
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_simmary = pd.DataFrame({'missing_count': missing, "missing_%": missing_pct})
print('Columns with NaN missing values:')
missing_simmary[missing_simmary['missing_count'] > 0]

In [ ]:
# STATS19 sentinel -1 values (encoded as "unknown/not applicable" in the raw data)
numeric_cols = df_raw.select_dtypes(include="number").columns
sentinel_counts = {
    col: (df_raw[col] == CONFIG['sentinel_value']).sum()
    for col in numeric_cols
}
sentinel_df = pd.DataFrame.from_dict(sentinel_counts, orient="index", columns=["sentinel_-1_count"])
print('Columns containing sentinel -1 values:')
sentinel_df[sentinel_df['sentinel_-1_count'] > 0].sort_values('sentinel_-1_count', ascending=False)

In [ ]:
# Duplicate rows and duplicate collision_index entries
print(f'Fully duplicate rows: {df_raw.duplicated().sum()}')
print(f'Duplicate collision_index values: {df_raw["collision_index"].duplicated().sum()}')
print(f'\nYear range: {df_raw["collision_year"].min()} — {df_raw["collision_year"].max()}')

# 3. Visualize Raw Data

In [ ]:
fig, ax = plt.subplots(figsize=CONFIG['figsize_wide'])
sns.heatmap(
    df_raw.isnull(),
    yticklabels=False,
    cbar=True,
    cmap='viridis',
    ax=ax
)
ax.set_title("Missing Value Heatmap (Raw Data)", fontsize=14, fontweight="bold")
ax.set_xlabel("Columns")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()